In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from reportlab.pdfgen import canvas
from reportlab.platypus import Table, TableStyle
from reportlab.lib import colors
%matplotlib inline

In [16]:
df = pd.read_csv(
    "../data/mora_julio_agosto_3.csv",
    sep=";",
    decimal=",",
    encoding="utf-8-sig"
)
df_mororsos = pd.read_csv(
    "../data/deudas_anteriores_julio.csv",
    sep=";",
    decimal=",",
    encoding="utf-8-sig"
)

In [17]:
df['JULIO'] = df['JULIO'].fillna('')
df['AGOSTO'] = df['AGOSTO'].fillna('')
df['SEPTIEMBRE'] = df['SEPTIEMBRE'].fillna('')
df['OCTUBRE'] = df['OCTUBRE'].fillna('')
df['NOVIEMBRE'] = df['NOVIEMBRE'].fillna('')
df['DICIEMBRE'] = df['DICIEMBRE'].fillna('')
df['ENERO'] = df['ENERO'].fillna('')
df['FEBRERO'] = df['FEBRERO'].fillna('')
df['MARZO'] = df['MARZO'].fillna('')
df['ABRIL'] = df['ABRIL'].fillna('')
df['MAYO'] = df['MAYO'].fillna('')
df['JUNIO'] = df['JUNIO'].fillna('')
df['PISO'] = df['PISO'].astype('Int64')
df['3y4 dormi'] = df['3y4 dormi'].fillna('')
df['Morosos'] = df['Morosos'].fillna('')

In [37]:
df_mororsos['PISO'] = df_mororsos['PISO'].astype('Int64')
df_mororsos_sin_duplicados = pd.DataFrame({
    'LETRA': df_mororsos['LETRA'],
    'PISO': df_mororsos['PISO']
})
df_mororsos_sin_duplicados = df_mororsos_sin_duplicados.drop_duplicates()
df_mororsos.info()

<class 'pandas.DataFrame'>
RangeIndex: 33 entries, 0 to 32
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   LETRA   32 non-null     str    
 1   PISO    32 non-null     Int64  
 2   Mes     32 non-null     str    
 3   Monto   32 non-null     float64
 4   Pagado  0 non-null      float64
 5   Saldo   32 non-null     float64
dtypes: Int64(1), float64(3), str(2)
memory usage: 1.7 KB


In [30]:
def generateTable(data):
    table = Table(data)
    # Definimos los estilos que queremos aplicar.
    # La lista de estilos contiene tuplas que especifican qué hacer.
    style = TableStyle([
        # Estilo para la fila de encabezados (fila 0, todas las columnas)
        ('BACKGROUND', (0, 0), (-1, 0), colors.grey),          # Fondo gris
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),     # Texto blanco
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),       # Fuente en negrita
        ('FONTSIZE', (0, 0), (-1, 0), 12),                     # Tamaño de fuente
        ('ALIGN', (0, 0), (-1, 0), 'CENTER'),                  # Alineación centrada
    
        # Estilo para el resto de las filas (de la 1 en adelante)
        ('BACKGROUND', (0, 1), (-1, -1), colors.beige),        # Fondo beige
        ('FONTNAME', (0, 1), (-1, -1), 'Helvetica'),
        ('FONTSIZE', (0, 1), (-1, -1), 10),
        ('ALIGN', (2, 1), (-1, -1), 'CENTER'),                 # Centrar columnas de montos
    
        # Estilo para toda la tabla
        ('GRID', (0, 0), (-1, -1), 1, colors.black),           # Poner una cuadrícula negra
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),                # Alineación vertical media
    ])
    # Aplicamos los estilos a la tabla.
    table.setStyle(style)

In [31]:
def formato_notificacion_mora(floor, 
                    number_of_department, 
                    user_name, 
                    current_date,
                    montos_x_mes,
                    total_adeudado,
                    fecha_corte):
    building_name = "Edificio Miraflores Norte"
    city_date = "La Paz, "+current_date
    departatment = "Departamento "+str(floor)+str(number_of_department)
    font_bold_style = "Helvetica-Bold"
    font_normal_style = "Helvetica"
    font_normal_size = 11
    font_medium_size = 12
    font_big_size = 16
    interlineado = 580
    parrafo = [
        "Por medio de la presente, la Administración del "+building_name+" se dirige a usted",
        "para comunicarle que, de acuerdo con nuestros registros administrativos, el departamento",
        str(floor)+str(number_of_department)+" presenta un saldo pendiente correspondiente al pago de expensas comunes, con",
        "corte al "+fecha_corte
    ]
    parrafo2 = [
        "Por tal motivo, solicitamos regularizar el saldo pendiente a la brevedad posible, a fin de",
        "mantener al día las obligaciones correspondientes a su unidad y contribuir al adecuado",
        "funcionamiento y mantenimiento de las áreas y servicios comunes del edificio."
    ]
    parrafo3 = [
        "En caso de que el pago ya hubiera sido efectuado, agradeceremos presentar o enviar el",
        "correspondiente comprobante a la Administración para realizar la verificación y",
        "actualización de nuestros registros."
    ]
    
    # Título
    pdf.setFont(font_bold_style, font_big_size)
    pdf.drawCentredString(300, 780, building_name)
    
    # Subtítulo
    pdf.setFont(font_normal_style, font_normal_size)
    pdf.drawCentredString(300, 760, "ADMINISTRACIÓN")
    
    # Fecha
    pdf.drawString(370, 720, city_date)
    
    # Destinatario
    pdf.setFont(font_bold_style, font_normal_size)
    pdf.drawString(80, 680, "Señor(a):")
    
    pdf.setFont(font_normal_style, font_normal_size)
    pdf.drawString(80, 665, user_name)
    pdf.drawString(80, 650, departatment)
    pdf.drawString(80, 635, 'Presente.-')
    
    # Referencia
    pdf.setFont(font_bold_style, font_normal_size)
    pdf.drawString(80, 600, "REF.: NOTIFICACIÓN DE MORA POR EXPENSAS")
    
    # Texto
    pdf.setFont(font_normal_style, font_normal_size)

    for linea in parrafo:
        interlineado = interlineado - 15
        pdf.drawString(
            80,
            interlineado,
            linea
        )
    
    # Datos de la deuda
    interlineado = interlineado - 20
    pdf.setFont(font_bold_style, font_normal_size)
    pdf.drawString(80, interlineado, "DETALLE DE LA DEUDA")

    interlineado = interlineado - 20
    pdf.setFont(font_normal_style, font_normal_size)
    pdf.drawString(80, interlineado, "Períodos adeudados:")

    tabla = Table(montos_x_mes)
    tabla.setStyle(TableStyle([
        # dibuja una cuadricula desde la primera cuadricula arriba a la izquierda hasta la última fila y columna de color negro con grosor 1
        ("GRID", (0, 0), (-1, -1), 1, colors.black), 
        ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
        ("ALIGN", (1, 0), (-1, -1), "CENTER"),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("FONTNAME", (-1, 0), (-1, -1), "Helvetica-Bold"),
    ]))
    interlineado = interlineado - 50
    tabla.wrapOn(pdf, 500, 100)
    tabla.drawOn(pdf, 80, interlineado)

    interlineado = interlineado - 20
    pdf.setFont(font_bold_style, font_medium_size)
    pdf.drawString(80, interlineado, f"TOTAL ADEUDADO: Bs. {total_adeudado}")

    interlineado = interlineado - 20
    pdf.setFont(font_normal_style, font_normal_size)
    for linea2 in parrafo2:
        pdf.drawString(
            80,
            interlineado,
            linea2
        )
        interlineado = interlineado - 15
    interlineado = interlineado - 5

    for linea3 in parrafo3:
        pdf.drawString(
            80,
            interlineado,
            linea3
        )
        interlineado = interlineado - 15

    # Firma
    interlineado = interlineado - 20
    pdf.setFont("Helvetica", 11)
    pdf.drawString(80, interlineado, "Atentamente,")
    interlineado = interlineado - 80
    pdf.drawString(80, interlineado, "____________________________")
    interlineado = interlineado - 15
    pdf.drawString(80, interlineado, "Administrador")
    interlineado = interlineado - 15
    pdf.drawString(80, interlineado, "Edif. " + building_name)
    pdf.showPage()

In [32]:
def verifica_deudas_anteriores_julio(LETRA, PISO):
    pass
def es_deudor_moroso(LETRA, PISO):
    if ((df_mororsos_sin_duplicados["LETRA"] == LETRA) & (df_mororsos_sin_duplicados["PISO"] == PISO)).any():
        return True
    else:
        return False

In [41]:
pdf = canvas.Canvas("carta_mora11.pdf")
current_date = '18 de septiembre de 2026'
debe_julio = False
debe_agosto = False
meses_de_mora = 2
fecha_corte = "13 de spetiembre 2026"
for indice, fila in df.iterrows():
    esDeudorAntesJulio = False
    esDeudorDespuesJulio = False
    monto_adeudado = 0
    montos_x_mes = [['Concepto', 'Expensas']]
    if str(fila["Morosos"]).strip() == "x" and str(fila["escidal"]).strip() != "x":
        if es_deudor_moroso(fila["LETRA"], fila["PISO"]):
            esDeudorAntesJulio = True
            for indicem, filam in df_mororsos.iterrows():
                #print(filam["Mes "])
                if fila["LETRA"] == filam["LETRA"] and fila["PISO"] == filam["PISO"]:
                    monto_adeudado = monto_adeudado + filam["Saldo"]
                    monto_adeudado = round(monto_adeudado, 2)
                    montos_x_mes.append([str(filam["Mes "]), filam["Saldo"]])
                
    if str(fila["3y4 dormi"]).strip() != "x" and str(fila["Morosos"]).strip() != "x" and str(fila["escidal"]).strip() != "x":
        numero_meses_adeudados = 0
        for nombre_mes, monto in fila.iloc[3:14].items():
            if str(monto).strip() != "":
                numero_meses_adeudados = numero_meses_adeudados + 1
                monto_adeudado = monto_adeudado + monto
                montos_x_mes.append([nombre_mes, monto])
        
        if numero_meses_adeudados >= meses_de_mora:
            esDeudorDespuesJulio = True
            monto_adeudado = round(monto_adeudado, 2)
            montos_x_mes.append(['Total', monto_adeudado])
            print(f"{str(indice)} El copropieatario {fila["LETRA"]}{fila["PISO"]} - {fila["NOMBRE Y APELLIDO"]} tiene dos meses de duda, meses adeudados {str(numero_meses_adeudados)}")
    if esDeudorAntesJulio or esDeudorDespuesJulio:
        formato_notificacion_mora(fila["PISO"], fila["LETRA"], fila["NOMBRE Y APELLIDO"], current_date, [list(x) for x in zip(*montos_x_mes)], monto_adeudado, fecha_corte)
    
# Guardar
pdf.save()

2 El copropieatario C3 - JOSE MANUEL BUSTILLOS tiene dos meses de duda, meses adeudados 2
5 El copropieatario F3 - BRIGITTE VARGAS/SHAKIRA VIDAL tiene dos meses de duda, meses adeudados 2
7 El copropieatario B4 - MANUEL A. LIRA ORTIZ tiene dos meses de duda, meses adeudados 2
8 El copropieatario C4 - SIRLEY VALENCIA/ANGEL VALENCVIA SIRPA tiene dos meses de duda, meses adeudados 2
14 El copropieatario C5 - LUZ  TICONA QUISPE tiene dos meses de duda, meses adeudados 2
23 El copropieatario F6 - BEATRIZ GONZALES DURAN tiene dos meses de duda, meses adeudados 2
30 El copropieatario A8 -  MOISES CABRERA/ELIANA TORRES tiene dos meses de duda, meses adeudados 2
42 El copropieatario A10 - MARCELO USCAMAITA tiene dos meses de duda, meses adeudados 2
69 El copropieatario D14 - JAVIER RODRIGO OJEDA OICAMPO tiene dos meses de duda, meses adeudados 2
78 El copropieatario A16 - PABEL SAINZ E,/ERICK JARANDILLA tiene dos meses de duda, meses adeudados 2
98 El copropieatario B20 - RAQUEL GUTIERREZ FLORE